# FedEx-LoRA — All 4 Phases (Fixed + Checkpointed)
**PDC Course Project**

Run cells **top to bottom**. Results auto-save to Google Drive after each phase.
GPU (T4) required. Estimated total runtime: **~55–65 min**.


In [ ]:
# CELL 0 — Install packages & mount Drive (run first, alone)
!pip install -q "torchao>=0.16.0" sacrebleu rouge-score

from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/FedExLoRA_Results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will be saved to: {SAVE_DIR}")


## Shared Setup — run Cells 1–6 before any phase

In [ ]:
# CELL 1 — Imports & Config
import torch, numpy as np, matplotlib.pyplot as plt, warnings, logging, pickle
from copy import deepcopy
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    GPT2LMHeadModel, GPT2Tokenizer, logging as hf_logging
)
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import get_peft_model, LoraConfig, TaskType

warnings.filterwarnings('ignore')
hf_logging.set_verbosity_error()
logging.getLogger().setLevel(logging.ERROR)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CFG = {
    'bert_model'   : 'bert-base-uncased',
    'num_clients'  : 10,
    'num_rounds'   : 5,
    'local_epochs' : 1,
    'lora_rank'    : 8,
    'lora_alpha'   : 16,
    'batch_size'   : 32,
    'lr'           : 2e-4,
    'dir_alpha'    : 0.3,
    'max_len_bert' : 128,
    'num_labels'   : 2,
    'train_subset' : 10000,
    'gpt2_model'   : 'gpt2',
    'max_len_gpt2' : 128,
    'e2e_subset'   : 3000,
    'e2e_rounds'   : 3,
    'e2e_epochs'   : 1,
    'svd_ranks'    : [2, 4, 6, 8],
    'seed'         : 42,
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print("Config ready.")


In [ ]:
# CELL 2 — Dataset Loading & Dirichlet Partition

def load_sst2(model_name, max_length, subset=None, seed=42):
    ds  = load_dataset('glue', 'sst2')
    tok = AutoTokenizer.from_pretrained(model_name)
    def tokenize(batch):
        enc = tok(batch['sentence'], padding='max_length',
                  truncation=True, max_length=max_length)
        enc['labels'] = batch['label']
        return enc
    train = ds['train'].map(tokenize, batched=True,
                            remove_columns=['sentence','label','idx'])
    val   = ds['validation'].map(tokenize, batched=True,
                                 remove_columns=['sentence','label','idx'])
    train.set_format('torch'); val.set_format('torch')
    if subset:
        rng   = np.random.default_rng(seed)
        train = train.select(rng.choice(len(train), subset, replace=False).tolist())
    return train, val, tok


def dirichlet_partition(dataset, num_clients, alpha, seed=42):
    rng = np.random.default_rng(seed)
    try:
        labels      = np.array([dataset[i]['labels'].item() for i in range(len(dataset))])
        num_classes = len(np.unique(labels))
    except Exception:
        idx    = rng.permutation(len(dataset)).tolist()
        splits = [[] for _ in range(num_clients)]
        for i, j in enumerate(idx):
            splits[i % num_clients].append(j)
        return splits

    splits = [[] for _ in range(num_clients)]
    for cls in range(num_classes):
        idx = np.where(labels == cls)[0]; rng.shuffle(idx)
        props  = rng.dirichlet([alpha] * num_clients)
        counts = (props * len(idx)).astype(int)
        counts[-1] = len(idx) - counts[:-1].sum()
        start = 0
        for i, cnt in enumerate(counts):
            splits[i].extend(idx[start:start+cnt].tolist()); start += cnt
    for s in splits:
        rng.shuffle(s)
    return splits


print("Loading SST-2 ...")
sst2_train, sst2_val, bert_tok = load_sst2(
    CFG['bert_model'], CFG['max_len_bert'], CFG['train_subset'], CFG['seed'])
client_splits = dirichlet_partition(
    sst2_train, CFG['num_clients'], CFG['dir_alpha'], CFG['seed'])

print(f"Train: {len(sst2_train)}  |  Val: {len(sst2_val)}")
print(f"\nDirichlet partition (alpha={CFG['dir_alpha']}):")
print(f"{'Client':>8} {'Samples':>8} {'Status':>8}")
for i, idx in enumerate(client_splits):
    status = 'SKIP' if len(idx) == 0 else 'OK'
    print(f"{i+1:>8} {len(idx):>8} {status:>8}")


In [ ]:
# CELL 3 — Model Builder & LoRA Utilities

def build_bert_lora(cfg):
    m  = AutoModelForSequenceClassification.from_pretrained(
             cfg['bert_model'], num_labels=cfg['num_labels'])
    lc = LoraConfig(task_type=TaskType.SEQ_CLS, r=cfg['lora_rank'],
                    lora_alpha=cfg['lora_alpha'],
                    target_modules=['query','value'],
                    lora_dropout=0.1, bias='none')
    return get_peft_model(m, lc)


def build_gpt2_lora(cfg):
    m  = GPT2LMHeadModel.from_pretrained(cfg['gpt2_model'])
    lc = LoraConfig(task_type=TaskType.CAUSAL_LM, r=cfg['lora_rank'],
                    lora_alpha=cfg['lora_alpha'],
                    target_modules=['c_attn'],
                    lora_dropout=0.1, bias='none')
    return get_peft_model(m, lc)


def get_lora_params(model):
    params = {}
    for name, mod in model.named_modules():
        if (hasattr(mod,'lora_A') and hasattr(mod,'lora_B')
                and hasattr(mod,'base_layer') and 'default' in mod.lora_A):
            params[name] = (mod.lora_B['default'].weight.data.clone(),
                            mod.lora_A['default'].weight.data.clone(),
                            mod.scaling['default'])
    return params


def set_lora_params(model, params):
    for name, mod in model.named_modules():
        if name in params and hasattr(mod,'lora_A'):
            mod.lora_B['default'].weight.data.copy_(params[name][0])
            mod.lora_A['default'].weight.data.copy_(params[name][1])


def get_base_weights(model):
    w = {}
    for name, mod in model.named_modules():
        if (hasattr(mod,'lora_A') and hasattr(mod,'base_layer')
                and 'default' in mod.lora_A):
            w[name] = mod.base_layer.weight.data.clone()
    return w


def set_base_weights(model, weights):
    for name, mod in model.named_modules():
        if name in weights and hasattr(mod,'base_layer'):
            mod.base_layer.weight.data.copy_(weights[name])


# Quick check
_m = build_bert_lora(CFG).to(DEVICE)
_p = get_lora_params(_m)
print(f"BERT LoRA layers: {len(_p)}")
_m.print_trainable_parameters()
del _m; torch.cuda.empty_cache()

_m = build_gpt2_lora(CFG).to(DEVICE)
_p = get_lora_params(_m)
print(f"GPT-2 LoRA layers: {len(_p)}")
_m.print_trainable_parameters()
del _m; torch.cuda.empty_cache()


In [ ]:
# CELL 4 — Aggregation Functions
# FIX: GPT-2 uses Conv1D whose weight is (in, out) — transposed vs nn.Linear.
#      We detect shape mismatch and transpose the LoRA product to match.

def _align(delta, base_w):
    """Transpose delta if shapes don't match base weight (Conv1D fix)."""
    if delta.shape == base_w.shape:
        return delta
    if delta.T.shape == base_w.shape:
        return delta.T
    raise ValueError(f"Cannot align delta {delta.shape} with base {base_w.shape}")


def fedavg_aggregate(client_params):
    k, out = len(client_params), {}
    for key in client_params[0]:
        B = sum(p[key][0] for p in client_params) / k
        A = sum(p[key][1] for p in client_params) / k
        out[key] = (B, A, client_params[0][key][2])
    return out


def fedex_aggregate(client_params, base_weights):
    k = len(client_params)
    lora_out, base_out = {}, {}
    for key in client_params[0]:
        Bs = [p[key][0] for p in client_params]
        As = [p[key][1] for p in client_params]
        sc     = client_params[0][key][2]
        B_avg  = sum(Bs) / k
        A_avg  = sum(As) / k
        BA_avg = sum(B @ A for B, A in zip(Bs, As)) / k
        delta  = _align((BA_avg - B_avg @ A_avg) * sc, base_weights[key])
        base_out[key] = base_weights[key] + delta
        lora_out[key] = (B_avg, A_avg, sc)
    return lora_out, base_out


def fedex_svd_aggregate(client_params, base_weights, svd_rank):
    k = len(client_params)
    lora_out, base_out = {}, {}
    for key in client_params[0]:
        Bs = [p[key][0] for p in client_params]
        As = [p[key][1] for p in client_params]
        sc     = client_params[0][key][2]
        B_avg  = sum(Bs) / k
        A_avg  = sum(As) / k
        BA_avg = sum(B @ A for B, A in zip(Bs, As)) / k
        delta  = (BA_avg - B_avg @ A_avg) * sc
        # Truncated SVD compression
        U, S, Vh = torch.linalg.svd(delta.float(), full_matrices=False)
        r_       = min(svd_rank, S.shape[0])
        delta_c  = (U[:, :r_] * S[:r_]) @ Vh[:r_, :]
        delta_c  = _align(delta_c.to(base_weights[key].dtype), base_weights[key])
        base_out[key] = base_weights[key] + delta_c
        lora_out[key] = (B_avg, A_avg, sc)
    return lora_out, base_out


def compute_bias(client_params, old_base, new_lora, new_base):
    k, total = len(client_params), 0.0
    for key in client_params[0]:
        Bs = [p[key][0] for p in client_params]
        As = [p[key][1] for p in client_params]
        sc       = client_params[0][key][2]
        BA_avg   = sum(B @ A for B, A in zip(Bs, As)) / k * sc
        BA_avg   = _align(BA_avg, old_base[key])           # Conv1D fix
        W_ideal  = old_base[key] + BA_avg
        lora_upd = _align(new_lora[key][0] @ new_lora[key][1] * sc, new_base[key])
        W_method = new_base[key] + lora_upd
        total   += torch.norm(W_ideal - W_method).item()
    return total


print("Aggregation functions defined (Conv1D-safe).")


In [ ]:
# CELL 5 — Unit Test

def unit_test(num_clients=10, r=8, d=256):
    W0 = torch.randn(d, d, device=DEVICE); sc = 2.0
    Bs = [torch.randn(d, r, device=DEVICE) for _ in range(num_clients)]
    As = [torch.randn(r, d, device=DEVICE) for _ in range(num_clients)]
    W_ideal  = W0 + sum(B@A for B,A in zip(Bs,As)) / num_clients * sc
    B_avg = sum(Bs)/num_clients; A_avg = sum(As)/num_clients
    W_fedavg = W0 + B_avg @ A_avg * sc
    BA_avg   = sum(B@A for B,A in zip(Bs,As)) / num_clients
    delta    = (BA_avg - B_avg @ A_avg) * sc
    W_fedex  = W0 + B_avg @ A_avg * sc + delta
    e_avg = torch.norm(W_ideal - W_fedavg).item()
    e_ex  = torch.norm(W_ideal - W_fedex).item()
    print("=" * 52)
    print("UNIT TEST — FedEx Exactness")
    print(f"  ||W_ideal - W_FedAvg||_F = {e_avg:>10.4f}  (bias exists)")
    print(f"  ||W_ideal - W_FedEx ||_F = {e_ex:>10.2e}  (should be ~0)")
    print(f"  Result: {'PASSED' if e_ex < 1e-4 else 'FAILED'}")
    print("=" * 52)

unit_test()


In [ ]:
# CELL 6 — Training & Evaluation Helpers

def train_client(model, data, cfg):
    if len(data) == 0:
        return get_lora_params(model), 0.0
    loader = DataLoader(data, batch_size=cfg['batch_size'],
                        shuffle=True, pin_memory=True, drop_last=False)
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=cfg['lr'])
    model.train(); total, steps = 0.0, 0
    for _ in range(cfg['local_epochs']):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss  = model(**batch).loss
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); steps += 1
    return get_lora_params(model), (total/steps if steps else 0.0)


def evaluate_cls(model, val_data, batch_size=64):
    loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, pin_memory=True)
    model.eval(); correct = total = 0
    with torch.no_grad():
        for batch in loader:
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            preds   = model(**batch).logits.argmax(-1)
            correct += (preds == batch['labels']).sum().item()
            total   += len(batch['labels'])
    return correct / total


def evaluate_lm(model, val_data, batch_size=16):
    loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, pin_memory=True)
    model.eval(); total_loss = total_tok = 0
    with torch.no_grad():
        for batch in loader:
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            out     = model(**batch)
            mask    = batch['labels'] != -100
            total_loss += out.loss.item() * mask.sum().item()
            total_tok  += mask.sum().item()
    avg = total_loss / total_tok if total_tok else float('inf')
    return avg   # return loss (lower = better)


def run_federated(method, cfg, splits, train_data, val_data,
                  eval_fn=None, svd_rank=None, lora_builder=None):
    label = {'fedavg':'FedAvg-LoRA (Biased)',
             'fedex' :'FedEx-LoRA  (Exact)',
             'fedex_svd':f"FedEx-SVD r'={svd_rank}"}[method]
    print(f"\n{'='*52}\n  {label}\n{'='*52}")

    if lora_builder is None: lora_builder = build_bert_lora
    if eval_fn is None:      eval_fn = lambda m: evaluate_cls(m, val_data)

    model  = lora_builder(cfg).to(DEVICE)
    accs, biases, res_norms = [], [], []
    active = [(i, splits[i]) for i in range(cfg['num_clients']) if len(splits[i]) > 0]

    for rnd in range(1, cfg['num_rounds'] + 1):
        g_lora = get_lora_params(model)
        g_base = get_base_weights(model)
        all_params, losses = [], []

        for cid, split in active:
            cm     = deepcopy(model)
            params, loss = train_client(cm, train_data.select(split), cfg)
            all_params.append(params); losses.append(loss)
            del cm; torch.cuda.empty_cache()

        if method == 'fedavg':
            new_lora = fedavg_aggregate(all_params); new_base = g_base
            set_lora_params(model, new_lora)
        elif method == 'fedex':
            new_lora, new_base = fedex_aggregate(all_params, g_base)
            set_lora_params(model, new_lora); set_base_weights(model, new_base)
        elif method == 'fedex_svd':
            new_lora, new_base = fedex_svd_aggregate(all_params, g_base, svd_rank)
            set_lora_params(model, new_lora); set_base_weights(model, new_base)

        bias   = compute_bias(all_params, g_base, new_lora, new_base)
        r_norm = 0.0
        for key in all_params[0]:
            Bs = [p[key][0] for p in all_params]
            As = [p[key][1] for p in all_params]
            sc = all_params[0][key][2]
            BA_avg = sum(B@A for B,A in zip(Bs,As)) / len(all_params)
            B_avg  = sum(Bs)/len(all_params); A_avg = sum(As)/len(all_params)
            r_norm += torch.norm((BA_avg - B_avg@A_avg)*sc).item()

        metric = eval_fn(model)
        accs.append(metric); biases.append(bias); res_norms.append(r_norm)
        print(f"  Round {rnd}: loss={sum(losses)/len(losses):.4f}  "
              f"metric={metric:.4f}  bias={bias:.3e}")

    return accs, biases, res_norms


def save_results(name, data):
    path = f'{SAVE_DIR}/{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f"Saved: {path}")

def save_plot(name):
    path = f'{SAVE_DIR}/{name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.savefig(f'/content/{name}.png', dpi=150, bbox_inches='tight')
    print(f"Saved: {path}")

print("All helpers defined.")


## Phase 1 + 2 — FedAvg-LoRA vs FedEx-LoRA on GLUE SST-2
Expected: ~15 min

In [ ]:
# CELL 7 — [Phase 1+2] Training

fedavg_accs, fedavg_biases, _ = run_federated(
    'fedavg', CFG, client_splits, sst2_train, sst2_val)

fedex_accs, fedex_biases, fedex_res = run_federated(
    'fedex', CFG, client_splits, sst2_train, sst2_val)

save_results('phase12', {
    'fedavg_accs': fedavg_accs, 'fedavg_biases': fedavg_biases,
    'fedex_accs' : fedex_accs,  'fedex_biases' : fedex_biases,
    'fedex_res'  : fedex_res})
print("Phase 1+2 COMPLETE")


In [ ]:
# CELL 8 — [Phase 1+2] Plots

rounds = list(range(1, CFG['num_rounds']+1))
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle(f"Phase 1+2 | GLUE SST-2 | BERT-base | "
             f"{CFG['num_clients']} clients | Dir a={CFG['dir_alpha']}",
             fontsize=13, fontweight='bold')

axes[0].plot(rounds, fedavg_accs, 'r-o', lw=2, ms=8, label='FedAvg-LoRA (Biased)')
axes[0].plot(rounds, fedex_accs,  'g-o', lw=2, ms=8, label='FedEx-LoRA  (Exact)')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Accuracy per Round'); axes[0].legend()
axes[0].grid(alpha=0.3); axes[0].set_ylim([0.5,1.0]); axes[0].set_xticks(rounds)

axes[1].plot(rounds, fedavg_biases, 'r-o', lw=2, ms=8, label='FedAvg-LoRA')
axes[1].plot(rounds, fedex_biases,  'g-o', lw=2, ms=8, label='FedEx-LoRA')
axes[1].set_xlabel('Round'); axes[1].set_ylabel('||W_ideal - W||_F')
axes[1].set_title('Aggregation Bias (Lower=Better)'); axes[1].legend()
axes[1].grid(alpha=0.3); axes[1].set_xticks(rounds)

plt.tight_layout(); save_plot('phase12_results'); plt.show()

print(f"\n{'='*56}")
print(f"{'Round':>6}  {'FedAvg':>10}  {'FedEx':>10}  {'Gap':>8}")
print("="*56)
for i,r in enumerate(rounds):
    print(f"{r:>6}  {fedavg_accs[i]:>10.4f}  {fedex_accs[i]:>10.4f}  "
          f"{fedex_accs[i]-fedavg_accs[i]:>+8.4f}")
print(f"\nFedEx advantage: {fedex_accs[-1]-fedavg_accs[-1]:+.4f} "
      f"({(fedex_accs[-1]-fedavg_accs[-1])*100:+.2f}%)")


## Phase 3a — Local Epochs Sweep  E ∈ {1, 3, 5}
Expected: ~15 min

In [ ]:
# CELL 9 — [Phase 3a] Local Epochs Sweep

EPOCHS_SWEEP   = [1, 3, 5]
epoch_results  = {}

for E in EPOCHS_SWEEP:
    cfg_e = {**CFG, 'local_epochs': E, 'num_rounds': 3}
    print(f"\n--- E={E} ---")
    accs, biases, norms = run_federated(
        'fedex', cfg_e, client_splits, sst2_train, sst2_val)
    epoch_results[E] = (accs, biases, norms)

save_results('phase3a', epoch_results)
print("\nPhase 3a COMPLETE")


In [ ]:
# CELL 10 — [Phase 3a] Plots

rounds3 = list(range(1, 4))
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle("Phase 3a — Synchronisation Frequency (FedEx-LoRA, SST-2)",
             fontsize=13, fontweight='bold')
colors = ['blue','orange','purple']

for (E, (accs, biases, norms)), col in zip(epoch_results.items(), colors):
    axes[0].plot(rounds3, accs,  '-o', lw=2, ms=8, color=col, label=f'E={E}')
    axes[1].plot(rounds3, norms, '-o', lw=2, ms=8, color=col, label=f'E={E}')

for ax, ylabel, title in zip(axes,
    ['Validation Accuracy', 'Sum ||delta_W_res||_F'],
    ['Accuracy vs Local Epochs', 'Residual Norm vs Local Epochs\n(higher E = larger residual)']):
    ax.set_xlabel('Round'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3); ax.set_xticks(rounds3)

plt.tight_layout(); save_plot('phase3a_epochs'); plt.show()

print("\nFinal accuracy per epoch setting:")
for E, (accs,_,_) in epoch_results.items():
    print(f"  E={E}: {accs[-1]:.4f}")


## Phase 3b — GPT-2 + E2E NLG Federated Fine-Tuning
Expected: ~15 min

In [ ]:
# CELL 11 — [Phase 3b] GPT-2 + E2E NLG Data Setup

gpt2_tok = GPT2Tokenizer.from_pretrained(CFG['gpt2_model'])
gpt2_tok.pad_token = gpt2_tok.eos_token

def tokenize_e2e(batch):
    texts = [f"Input: {mr}\nOutput: {ref}{gpt2_tok.eos_token}"
             for mr, ref in zip(batch['meaning_representation'],
                                batch['human_reference'])]
    enc    = gpt2_tok(texts, padding='max_length', truncation=True,
                      max_length=CFG['max_len_gpt2'])
    labels = [ids[:] for ids in enc['input_ids']]
    for i, mask in enumerate(enc['attention_mask']):
        for j, m in enumerate(mask):
            if m == 0: labels[i][j] = -100
    enc['labels'] = labels
    return enc

print("Loading E2E NLG ...")
e2e_raw   = load_dataset('e2e_nlg')
e2e_train = e2e_raw['train'].map(
    tokenize_e2e, batched=True, remove_columns=e2e_raw['train'].column_names)
e2e_val   = e2e_raw['validation'].map(
    tokenize_e2e, batched=True, remove_columns=e2e_raw['validation'].column_names)
e2e_train.set_format('torch'); e2e_val.set_format('torch')

rng       = np.random.default_rng(CFG['seed'])
e2e_train = e2e_train.select(
    rng.choice(len(e2e_train), min(CFG['e2e_subset'], len(e2e_train)),
               replace=False).tolist())
e2e_splits = dirichlet_partition(e2e_train, CFG['num_clients'],
                                 CFG['dir_alpha'], seed=CFG['seed'])
print(f"E2E train: {len(e2e_train)}  |  Val: {len(e2e_val)}")

_gm = build_gpt2_lora(CFG).to(DEVICE)
_gm.print_trainable_parameters()
del _gm; torch.cuda.empty_cache()


In [ ]:
# CELL 12 — [Phase 3b] E2E Federated Training

cfg_e2e  = {**CFG, 'num_rounds': CFG['e2e_rounds'],
            'local_epochs': CFG['e2e_epochs'],
            'batch_size': 16, 'lr': 5e-5}
eval_e2e = lambda m: evaluate_lm(m, e2e_val, batch_size=16)

print("E2E FedAvg-LoRA ...")
e2e_fedavg_loss, e2e_fedavg_bias, _ = run_federated(
    'fedavg', cfg_e2e, e2e_splits, e2e_train, e2e_val,
    eval_fn=eval_e2e, lora_builder=build_gpt2_lora)

print("\nE2E FedEx-LoRA ...")
e2e_fedex_loss, e2e_fedex_bias, _ = run_federated(
    'fedex', cfg_e2e, e2e_splits, e2e_train, e2e_val,
    eval_fn=eval_e2e, lora_builder=build_gpt2_lora)

save_results('phase3b', {
    'fedavg_loss': e2e_fedavg_loss, 'fedavg_bias': e2e_fedavg_bias,
    'fedex_loss' : e2e_fedex_loss,  'fedex_bias' : e2e_fedex_bias})
print("\nPhase 3b COMPLETE")


In [ ]:
# CELL 13 — [Phase 3b] E2E Results Plot

rounds_e2e = list(range(1, CFG['e2e_rounds']+1))
fig, axes  = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle("Phase 3b — GPT-2 Federated Fine-Tuning on E2E NLG",
             fontsize=13, fontweight='bold')

axes[0].plot(rounds_e2e, e2e_fedavg_loss, 'r-o', lw=2, ms=8, label='FedAvg-LoRA')
axes[0].plot(rounds_e2e, e2e_fedex_loss,  'g-o', lw=2, ms=8, label='FedEx-LoRA')
axes[0].set_xlabel('Round'); axes[0].set_ylabel('Validation Loss (lower=better)')
axes[0].set_title('E2E NLG Validation Loss'); axes[0].legend()
axes[0].grid(alpha=0.3); axes[0].set_xticks(rounds_e2e)

axes[1].plot(rounds_e2e, e2e_fedavg_bias, 'r-o', lw=2, ms=8, label='FedAvg-LoRA Bias')
axes[1].plot(rounds_e2e, e2e_fedex_bias,  'g-o', lw=2, ms=8, label='FedEx-LoRA Bias')
axes[1].set_xlabel('Round'); axes[1].set_ylabel('Aggregation Bias ||W_ideal - W||_F')
axes[1].set_title('Aggregation Bias'); axes[1].legend()
axes[1].grid(alpha=0.3); axes[1].set_xticks(rounds_e2e)

plt.tight_layout(); save_plot('phase3b_e2e'); plt.show()

print(f"\nFinal Val Loss — FedAvg: {e2e_fedavg_loss[-1]:.4f}  "
      f"FedEx: {e2e_fedex_loss[-1]:.4f}  "
      f"Improvement: {e2e_fedavg_loss[-1]-e2e_fedex_loss[-1]:+.4f}")


## Phase 4 — SVD Residual Compression (Novel Extension)
Sweeps r' ∈ {2, 4, 6, 8}. Expected: ~12 min

In [ ]:
# CELL 14 — [Phase 4] SVD Compression Sweep

cfg_svd     = {**CFG, 'num_rounds': 3}
svd_results = {}

for r_ in CFG['svd_ranks']:
    print(f"\n--- FedEx-SVD  r'={r_} ({r_*100//CFG['lora_rank']}% of r) ---")
    accs, biases, norms = run_federated(
        'fedex_svd', cfg_svd, client_splits, sst2_train, sst2_val, svd_rank=r_)
    svd_results[r_] = (accs, biases, norms)

save_results('phase4', {
    'svd_results'     : svd_results,
    'fedavg_final_acc': fedavg_accs[-1],
    'fedex_final_acc' : fedex_accs[-1]})
print("\nPhase 4 COMPLETE")


In [ ]:
# CELL 15 — [Phase 4] Pareto Frontier Plot

def comm_params(method, svd_rank=None, d=768, r=8, k=10, n=24):
    ab = 2 * r * d
    if method == 'fedavg':    per = ab
    elif method == 'fedex':   per = ab + d * d
    else:                     per = ab + svd_rank * (2*d + 1)
    return k * per * n

rounds_svd    = list(range(1, cfg_svd['num_rounds']+1))
svd_accs_fin  = [svd_results[r_][0][-1] for r_ in CFG['svd_ranks']]
svd_costs     = [comm_params('svd', r_) for r_ in CFG['svd_ranks']]
fedavg_cost   = comm_params('fedavg')
fedex_cost    = comm_params('fedex')
colors_svd    = ['#1f77b4','#ff7f0e','#2ca02c','#9467bd']

fig, axes = plt.subplots(1, 2, figsize=(15,6))
fig.suptitle("Phase 4 — SVD Residual Compression (FedEx-LoRA, GLUE SST-2)",
             fontsize=13, fontweight='bold')

# Left: accuracy per round
ax = axes[0]
ax.axhline(fedavg_accs[-1], color='red',   ls='--', lw=1.5,
           label=f"FedAvg baseline ({fedavg_accs[-1]:.3f})")
ax.axhline(fedex_accs[-1],  color='green', ls='--', lw=1.5,
           label=f"FedEx full ({fedex_accs[-1]:.3f})")
for (r_,(accs,_,_)), col in zip(svd_results.items(), colors_svd):
    ax.plot(rounds_svd, accs, '-o', lw=2, ms=8, color=col,
            label=f"SVD r'={r_} ({r_*100//8}%)")
ax.set_xlabel('Round'); ax.set_ylabel('Validation Accuracy')
ax.set_title('Accuracy per Round (per SVD rank)')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xticks(rounds_svd)

# Right: Pareto frontier
ax = axes[1]
ax.scatter([fedavg_cost],[fedavg_accs[-1]], color='red',  s=180, zorder=5,
           marker='X', label='FedAvg-LoRA')
ax.scatter([fedex_cost], [fedex_accs[-1]],  color='green',s=180, zorder=5,
           marker='X', label='FedEx-LoRA (full residual)')
for r_, acc, cost, col in zip(CFG['svd_ranks'], svd_accs_fin, svd_costs, colors_svd):
    ax.scatter([cost],[acc], color=col, s=130, zorder=5,
               label=f"SVD r'={r_} ({100-cost*100//fedex_cost}% saving)")
ax.plot(svd_costs, svd_accs_fin, 'k--', lw=1, alpha=0.4)
ax.set_xlabel('Comm. Cost (# float32 params / round)')
ax.set_ylabel('Final Validation Accuracy')
ax.set_title('Pareto Frontier\nAccuracy vs Communication Cost')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))

plt.tight_layout(); save_plot('phase4_pareto'); plt.show()

print(f"\n{'='*65}")
print(f"{'Method':<24} {'Acc':>8} {'Cost (M)':>10} {'Saving vs FedEx':>16}")
print("="*65)
print(f"{'FedAvg-LoRA':<24} {fedavg_accs[-1]:>8.4f} {fedavg_cost/1e6:>10.2f} {'N/A':>16}")
for r_, acc, cost in zip(CFG['svd_ranks'], svd_accs_fin, svd_costs):
    saving = (1 - cost/fedex_cost)*100
    print(f"{'FedEx-SVD r='+str(r_):<24} {acc:>8.4f} {cost/1e6:>10.2f} {saving:>15.1f}%")
print(f"{'FedEx-LoRA (full)':<24} {fedex_accs[-1]:>8.4f} {fedex_cost/1e6:>10.2f} {'0%':>16}")
print("="*65)
print(f"\nAll results saved to: {SAVE_DIR}")
print("All 4 phases complete.")
